In [1]:
import pandas as pd

hotel = pd.read_csv("../data/hotel_bookings.csv")


In [2]:
manquants = hotel.isna().sum()
part = (manquants / len(hotel) * 100).round(1)
tableau = pd.DataFrame({"nb_vides": manquants, "pourcentage": part})

print(tableau[tableau["nb_vides"] > 0].sort_values("nb_vides", ascending=False))
hotel_propre = hotel.drop(columns=["company"])
print(hotel_propre.shape , hotel.shape)


          nb_vides  pourcentage
company     112593         94.3
agent        16340         13.7
country        488          0.4
children         4          0.0
(119390, 31) (119390, 32)


## Journal de nettoyage

### 1. Colonne `company` (94,3 % de valeurs manquantes)

- **Constat :** 112 593 lignes sur 119 390 n'ont pas de valeur pour `company`.
- **Décision :** suppression de la colonne.
- **Justification :** remplir ces lignes reviendrait à inventer presque toute
  la colonne. Aucune des cinq questions du projet n'utilise le numéro de
  l'entreprise.
- **Vérification :** le tableau passe de 32 à 31 colonnes, avec toujours
  119 390 lignes.

In [3]:
hotel_propre["agent"]= hotel_propre["agent"].fillna(0).astype(int)
print(hotel_propre["agent"].isna().sum())
print(hotel_propre["agent"].dtypes)

0
int64


### 2. Colonne `agent` (13,7 % de valeurs manquantes)

- **Constat :** 16 340 lignes sur 119 390 n'ont pas de numéro d'agence.
  La colonne est lue en nombres à virgule à cause de ces cases vides.
- **Décision :** conserver la colonne, remplacer les cases vides par `0`
  (« sans agence ») et convertir en entiers.
- **Justification :** 86 % des lignes ont un vrai numéro, donc la colonne reste
  utile. Le numéro `0` n'est utilisé par aucune agence (0 ligne), il ne se
  confond donc pas avec une vraie valeur.
- **Vérification :** plus aucune case vide dans `agent`, et son type est
  maintenant `int64`.

In [4]:
hotel_propre["country"]= hotel_propre["country"].fillna("Inconnue")
print(hotel_propre["country"].isna().sum())
print((hotel_propre["country"]== "Inconnue").sum())

0
488


### 3. Colonne `country` (0,4 % de valeurs manquantes)

- **Constat :** 488 lignes sur 119 390 n'ont pas de pays.
- **Décision :** conserver ces lignes et remplacer les cases vides par le
  texte « Inconnue ».
- **Justification :** retirer ces lignes ferait perdre 488 réservations
  complètes pour les autres questions (annulations, prix, délai, segment).
  « Inconnue » reste une catégorie distincte des vrais pays.
- **Vérification :** plus aucune case vide dans `country`, et 488 lignes valent
  « Inconnue ».

In [5]:
hotel_propre = hotel_propre.dropna(subset=["children"])
hotel_propre["children"] = hotel_propre["children"].astype(int)
print(len(hotel_propre["children"]))

119386


### 4. Colonne `children` (4 valeurs manquantes)

- **Constat :** 4 lignes sur 119 390 n'ont pas de nombre d'enfants. La colonne
  est lue en nombres à virgule à cause de ces cases vides.
- **Décision :** retrait des 4 lignes, puis conversion de la colonne en
  entiers.
- **Justification :** 4 lignes représentent moins de 0,01 % du jeu, donc la
  perte est négligeable. Remplacer par 0 aurait supposé, sans preuve, que ces
  réservations n'avaient pas d'enfants.
- **Vérification :** le tableau passe de 119 390 à 119 386 lignes, et
  `children` est de type `int64`.

In [6]:
hotel_propre["reservation_status_date"]= pd.to_datetime(hotel_propre["reservation_status_date"])
print(hotel_propre["reservation_status_date"].min())
print(hotel_propre["reservation_status_date"].max())

2014-10-17 00:00:00
2017-09-14 00:00:00


### 5. Colonne `reservation_status_date` (dates lues comme du texte)

- **Constat :** la colonne est lue comme du texte, donc on ne peut ni extraire
  le mois ni calculer d'écart.
- **Décision :** conversion en vraies dates avec `pd.to_datetime`.
- **Justification :** c'est nécessaire pour toute analyse temporelle.
- **Vérification :** la colonne est de type `datetime64`, et ses dates vont du
  17/10/2014 au 14/09/2017. Cette date correspond au dernier état de la
  réservation, par exemple le jour d'une annulation, d'où une première date
  antérieure aux premières arrivées (2015).

In [7]:
numero_mois = { 
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12,
}
hotel_propre["mois_arrivee"]= hotel_propre["arrival_date_month"].map(numero_mois)
print(hotel_propre["mois_arrivee"])
hotel_propre["date_arrivee"] = pd.to_datetime(
    {
        "year": hotel_propre["arrival_date_year"],
        "month": hotel_propre["mois_arrivee"],
        "day": hotel_propre["arrival_date_day_of_month"],
    }
)
print(hotel_propre["date_arrivee"].head())
print(hotel_propre["date_arrivee"].min(), hotel_propre["date_arrivee"].max())


0         7
1         7
2         7
3         7
4         7
         ..
119385    8
119386    8
119387    8
119388    8
119389    8
Name: mois_arrivee, Length: 119386, dtype: int64
0   2015-07-01
1   2015-07-01
2   2015-07-01
3   2015-07-01
4   2015-07-01
Name: date_arrivee, dtype: datetime64[us]
2015-07-01 00:00:00 2017-08-31 00:00:00


### 6. Date d'arrivée éclatée en trois colonnes

- **Constat :** l'arrivée est répartie sur `arrival_date_year`,
  `arrival_date_month` (en anglais, en toutes lettres) et
  `arrival_date_day_of_month`. On ne peut pas filtrer ni regrouper par mois ou
  par semaine directement.
- **Décision :** traduction du mois en numéro (colonne `mois_arrivee`), puis
  création d'une colonne `date_arrivee` de type date.
- **Justification :** c'est nécessaire pour l'analyse du temps et pour la
  question sur les mois les plus chargés.
- **Vérification :** les dates d'arrivée vont du 01/07/2015 au 31/08/2017.

In [8]:

print(hotel_propre[["adr", "adults", "lead_time", "stays_in_week_nights"]].describe())



                 adr         adults      lead_time  stays_in_week_nights
count  119386.000000  119386.000000  119386.000000         119386.000000
mean      101.833541       1.856390     104.014801              2.500310
std        50.534664       0.579261     106.863286              1.908289
min        -6.380000       0.000000       0.000000              0.000000
25%        69.290000       2.000000      18.000000              1.000000
50%        94.590000       2.000000      69.000000              2.000000
75%       126.000000       2.000000     160.000000              3.000000
max      5400.000000      55.000000     737.000000             50.000000


In [9]:

avant = len(hotel_propre)
hotel_propre = hotel_propre[hotel_propre["adr"] >= 0]
print(avant, len(hotel_propre), avant - len(hotel_propre))

119386 119385 1


### 7. Colonne `adr` : prix par nuit négatif

- **Constat :** une ligne a un prix par nuit négatif (-6,38 €). Un prix par
  nuit ne peut pas être négatif.
- **Décision :** retrait de cette ligne.
- **Justification :** une seule ligne sur 119 386, donc la perte est
  négligeable, et la valeur est impossible dans le contexte.
- **Vérification :** le tableau passe de 119 386 à 119 385 lignes, et le
  minimum de `adr` n'est plus négatif.

In [10]:
print(hotel_propre["adr"].sort_values(ascending=False).head())
avant = len(hotel_propre)
hotel_propre = hotel_propre[hotel_propre["adr"] < 1000]
print(avant, len(hotel_propre), avant - len(hotel_propre))

48515     5400.0
111403     510.0
15083      508.0
103912     451.5
13142      450.0
Name: adr, dtype: float64


119385 119384 1


In [11]:
print(hotel_propre[["adr", "adults", "lead_time", "stays_in_week_nights"]].describe())

                 adr         adults      lead_time  stays_in_week_nights
count  119384.000000  119384.000000  119384.000000         119384.000000
mean      101.790068       1.856388     104.014617              2.500293
std        48.151476       0.579265     106.863670              1.908273
min         0.000000       0.000000       0.000000              0.000000
25%        69.290000       2.000000      18.000000              1.000000
50%        94.590000       2.000000      69.000000              2.000000
75%       126.000000       2.000000     160.000000              3.000000
max       510.000000      55.000000     737.000000             50.000000


### 8. Colonne `adr` : prix aberrant de 5 400 €

- **Constat :** un prix par nuit de 5 400 € est isolé. Les prix suivants sont
  510, 508, 451,5 et 450 €. Il vaut environ 57 fois la médiane (94,59 €) et
  dépasse largement la borne des valeurs extrêmes (environ 211 €).
- **Décision :** retrait de cette ligne, avec un seuil à 1 000 € (tout ce qui
  est inférieur est conservé).
- **Justification :** ce prix est isolé et sans commune mesure avec le reste,
  donc très probablement une erreur de saisie. Il fausserait les moyennes de
  prix. Les prix de 450 à 510 € sont cohérents entre eux et sont conservés.
- **Vérification :** le tableau passe de 119 385 à 119 384 lignes, et le
  maximum de `adr` vaut maintenant 510 €.

In [12]:
sans_adulte = hotel_propre[hotel_propre["adults"] == 0]
print(len(sans_adulte))

403


In [13]:
vides = hotel_propre[
    (hotel_propre["adults"] == 0)
    & (hotel_propre["children"] == 0)
    & (hotel_propre["babies"] == 0)
]
len(vides)

180

In [14]:

avant = len(hotel_propre)
hotel_propre = hotel_propre.drop(vides.index)
print(avant, len(hotel_propre), avant - len(hotel_propre))

119384 119204 180


### 9. Réservations vides (0 adulte, 0 enfant, 0 bébé)

- **Constat :** 403 réservations ont `adults = 0`. Parmi elles, 180 n'ont
  aussi aucun enfant ni aucun bébé : personne n'est attendu à l'hôtel.
- **Décision :** retrait de ces 180 lignes.
- **Justification :** une réservation sans aucun client n'a pas de sens, c'est
  très probablement une erreur de saisie. Elles représentent environ 0,15 % du
  jeu, donc la perte est négligeable.
- **Vérification :** le tableau passe de 119 384 à 119 204 lignes. Les 223
  autres réservations sans adulte sont traitées au point 10.

### 10. Réservations sans adulte mais avec enfants ou bébés

- **Constat :** 223 réservations ont `adults = 0` mais au moins un enfant ou
  un bébé.
- **Décision :** conservation de ces lignes, sans modification.
- **Justification :** elles contiennent de vraies informations (prix, délai,
  statut d'annulation, pays) et les questions du projet n'utilisent pas le
  nombre d'adultes. Le nombre d'adultes est peut-être faussé pour ces lignes ;
  ce point est signalé dans les limites.
- **Vérification :** aucune ligne retirée, le tableau garde 119 204 lignes.

In [15]:
print(hotel_propre.duplicated().sum())
sans_duplications = hotel_propre.drop_duplicates()
print(hotel_propre["is_canceled"].mean())
print(sans_duplications["is_canceled"].mean())


31987
0.370742592530452
0.27522157377575474


In [16]:
avant = len(hotel_propre)
hotel_propre = hotel_propre.drop_duplicates()
print(avant , len(hotel_propre))

119204 87217


### 11. Lignes en double

- **Constat :** 31 987 lignes (environ 27 %) sont identiques, sur toutes les
  colonnes, à une ligne plus haute. Le fichier n'a pas de numéro de
  réservation, donc on ne peut pas prouver qu'il s'agit de la même réservation.
- **Décision :** retrait des doublons (`drop_duplicates`).
- **Justification :** ces lignes sont annulées bien plus souvent que les
  autres (environ 63 % contre 27,5 %), ce qui suggère des réservations
  enregistrées plusieurs fois. Les garder ferait compter plusieurs fois la
  même réservation et gonflerait le taux d'annulation.
- **Impact mesuré :** le taux d'annulation passe de 37,1 % (avec doublons) à
  27,5 % (sans). Ce choix change donc nettement les résultats ; il est repris
  dans les limites.
- **Vérification :** le tableau passe de 119 204 à 87 217 lignes.

### Exploration : les séjours de 0 nuit

On compte les lignes sans nuit (week-end + semaine = 0) et on regarde leur
prix par nuit avant de décider quoi en faire.

In [17]:
nuits = hotel_propre["stays_in_weekend_nights"] + hotel_propre["stays_in_week_nights"]
print((nuits == 0).sum())
sans_nuit = hotel_propre[nuits == 0]
print(sans_nuit["adr"].describe())
print((sans_nuit["adr"] == 0).sum())

591
count    591.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: adr, dtype: float64
591


In [18]:
nuits = hotel_propre["stays_in_weekend_nights"] + hotel_propre["stays_in_week_nights"]
avant = len(hotel_propre)
hotel_propre = hotel_propre[nuits > 0]
print(avant, len(hotel_propre), avant - len(hotel_propre))

87217 86626 591


### 12. Séjours de 0 nuit

- **Constat :** 591 lignes ont 0 nuit en week-end et 0 nuit en semaine. Toutes
  ont un prix par nuit (`adr`) égal à 0.
- **Décision :** retrait de ces 591 lignes.
- **Justification :** sans nuit et sans prix, ce ne sont pas de vrais séjours
  payés. Elles fausseraient les moyennes de prix par nuit (questions 2 et 4).
- **Vérification :** le tableau passe de 87 217 à 86 626 lignes.

In [19]:
print(hotel_propre.shape)
print(hotel_propre.isna().sum().sum())
print(hotel_propre[["reservation_status_date", "date_arrivee", "children", "agent"]].dtypes)
print(hotel_propre["adr"].min(), hotel_propre["adr"].max())

(86626, 33)
0
reservation_status_date    datetime64[us]
date_arrivee               datetime64[us]
children                            int64
agent                               int64
dtype: object
0.0 510.0


In [20]:
hotel_propre.to_csv("../data/hotel_bookings_propre.csv", index=False)

## Bilan du nettoyage

| Étape | Lignes retirées | Lignes restantes |
| --- | --- | --- |
| Fichier brut | | 119 390 |
| `children` vide | 4 | 119 386 |
| Prix par nuit négatif | 1 | 119 385 |
| Prix aberrant (5 400 €) | 1 | 119 384 |
| Réservations vides | 180 | 119 204 |
| Doublons | 31 987 | 87 217 |
| Séjours de 0 nuit | 591 | 86 626 |

- **Colonnes :** 32 au départ ; `company` supprimée ; `mois_arrivee` et
  `date_arrivee` ajoutées, soit 33 colonnes.
- **Corrections sans retrait de lignes :** `agent` (0 = sans agence),
  `country` (« Inconnue »), `reservation_status_date` convertie en date.
- **Lignes conservées malgré une anomalie :** 223 réservations sans adulte
  mais avec enfants ou bébés.
- **Fichier produit :** `data/hotel_bookings_propre.csv`.